# QR Scan Consumer & Anomaly Detector

## Purpose of This Notebook

This notebook implements the **consumer and anomaly detection stage** of the cybersecurity pipeline.

Its responsibility is to:

- consume QR scan events from Kafka,
- apply the trained Isolation Forest model for anomaly detection,
- emit **distributed tracing information**,
- and persist detection results for later analysis.

This notebook represents the **core analytical stage** of the pipeline.

---

## Role in the Overall Pipeline

Conceptually, this notebook corresponds to the **anomaly detection layer** in a security pipeline.

In real-world systems, this stage would include:

- machine learning model inference,
- threshold-based detection,
- real-time alerting,
- enrichment with context.

Here, the logic uses the pre-trained Isolation Forest model.

---

## Key Architectural Ideas Demonstrated

### 1. Asynchronous Consumption from a Queue

The consumer reads events from Kafka using a **consumer group**.

Important implications:

- the consumer is decoupled from the producer,
- it can be restarted without data loss,
- multiple consumers could be added to scale processing.

### 2. Stateless Processing with Persistent Output

The consumer:

- does **not** store internal state,
- processes each event independently,
- writes results to a persistent output file.

This design makes the pipeline:

- easier to reason about,
- easier to scale,
- easier to debug.

### 3. Machine Learning Inference in Real-Time

Each event is processed through the trained Isolation Forest model:

- Features are preprocessed using saved scaler and encoder
- Anomaly scores are computed
- Binary predictions are made

---

## Tracing Model Used Here

This notebook continues the **end-to-end trace** started by the producer.

Key design choices:

- the `trace_id` is reconstructed from `event_id`,
- consumer spans become **children of producer spans**,
- each processing step is represented as a separate span.

As a result, one event appears in Jaeger as **one complete pipeline trace**.

---

## What Is Traced

For each consumed event, the following spans are emitted:

- `consume_qr_event` — message consumption from Kafka
- `preprocess_features` — feature preprocessing
- `detect_anomaly` — anomaly detection with ML model
- `write_tsv` — persistence of detection results

Each span contains relevant attributes like anomaly score and prediction.

---

## Output Format

Detection results are written to a local TSV file: `anomaly_detection_results.tsv`

This file acts as a **materialized view** of the pipeline output.

TSV is used because:

- it requires no database setup,
- it is easy to inspect,
- it integrates naturally with Jupyter for analysis.

---

## Model Loading

The notebook loads the pre-trained model and preprocessing objects:

- `isolation_forest_model.pkl` — Trained Isolation Forest
- `scaler.pkl` — Feature scaler
- `encoder.pkl` — Label encoder for protocol

In [ ]:
import json
import csv
import os
import random
import pickle
from datetime import datetime

from kafka import KafkaConsumer

# ---------------- OpenTelemetry ----------------
from opentelemetry import trace
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import (
    OTLPSpanExporter,
)
from opentelemetry.trace import SpanKind, TraceFlags
from opentelemetry.trace import set_span_in_context
from opentelemetry.trace import SpanContext, NonRecordingSpan

# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

KAFKA_BOOTSTRAP_SERVERS = "kafka:9092"
KAFKA_TOPIC = "qr-events"  # Different topic
KAFKA_GROUP_ID = "qr-anomaly-detector"  # Different group

OTLP_ENDPOINT = "jaeger:4317"
SERVICE_NAME = "qr-anomaly-detector"  # Different service

OUTPUT_TSV = "anomaly_detection_results.tsv"

# ---------------------------------------------------------------------
# Tracing setup (OTLP → Jaeger)
# ---------------------------------------------------------------------

resource = Resource.create(
    {
        "service.name": SERVICE_NAME,
    }
)

trace.set_tracer_provider(TracerProvider(resource=resource))
tracer = trace.get_tracer(__name__)

otlp_exporter = OTLPSpanExporter(
    endpoint=OTLP_ENDPOINT,
    insecure=True,
)

span_processor = BatchSpanProcessor(otlp_exporter)
trace.get_tracer_provider().add_span_processor(span_processor)

# ---------------------------------------------------------------------
# Load trained model and preprocessing objects
# ---------------------------------------------------------------------

print("Loading trained model and preprocessing objects...")

with open('isolation_forest_model.pkl', 'rb') as f:
    iso_forest = pickle.load(f)

with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

with open('encoder.pkl', 'rb') as f:
    encoder = pickle.load(f)

print("Model loaded successfully!")

# ---------------------------------------------------------------------
# Kafka consumer
# ---------------------------------------------------------------------

consumer = KafkaConsumer(
    KAFKA_TOPIC,
    bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
    group_id=KAFKA_GROUP_ID,
    auto_offset_reset="earliest",
    enable_auto_commit=True,
    value_deserializer=lambda v: json.loads(v.decode("utf-8")),
)

print("Kafka consumer started")
print(f"Topic: {KAFKA_TOPIC}")
print(f"OTLP endpoint: {OTLP_ENDPOINT}")

# ---------------------------------------------------------------------
# Anomaly detection function
# ---------------------------------------------------------------------

def detect_anomaly(event):
    """
    Apply the trained Isolation Forest model to detect anomalies.
    """
    
    # Extract features
    features = {
        'hour': event['hour'],
        'url_length': event['url_length'],
        'dot_count': event['dot_count'],
        'protocol': event['protocol']
    }
    
    # Preprocess features
    df_features = pd.DataFrame([features])
    df_features['protocol'] = encoder.transform(df_features['protocol'])
    scaled_features = scaler.transform(df_features)
    
    # Get anomaly score and prediction
    anomaly_score = iso_forest.decision_function(scaled_features)[0]
    prediction = iso_forest.predict(scaled_features)[0]
    
    # Convert prediction: -1 (anomaly) -> 1, 1 (normal) -> 0
    is_anomaly = 1 if prediction == -1 else 0
    
    return anomaly_score, is_anomaly

# ---------------------------------------------------------------------
# TSV initialization
# ---------------------------------------------------------------------

file_exists = os.path.exists(OUTPUT_TSV)

tsv_file = open(OUTPUT_TSV, "a", newline="")
fieldnames = [
    "event_id",
    "timestamp",
    "user",
    "device_id",
    "url",
    "url_length",
    "dot_count",
    "protocol",
    "hour",
    "anomaly_score",
    "is_anomaly_predicted",
    "is_anomaly_ground_truth"
]

writer = csv.DictWriter(tsv_file, fieldnames=fieldnames, delimiter='\t')

if not file_exists:
    writer.writeheader()

# ---------------------------------------------------------------------
# Main consume loop
# ---------------------------------------------------------------------

import pandas as pd  # For DataFrame in detect_anomaly

for msg in consumer:
    event = msg.value
    event_id = event["event_id"]

    # Restore trace context from event_id
    trace_id = int(event_id.replace("-", ""), 16)

    parent_ctx = set_span_in_context(
        NonRecordingSpan(
            SpanContext(
                trace_id=trace_id,
                span_id=random.getrandbits(64),
                is_remote=True,
                trace_flags=TraceFlags(TraceFlags.SAMPLED),
                trace_state={},
            )
        )
    )

    with tracer.start_as_current_span(
        "consume_qr_event",
        context=parent_ctx,
        kind=SpanKind.CONSUMER,
    ) as span:

        span.set_attribute("event.id", event_id)
        span.set_attribute("event.type", "qr_scan")
        span.set_attribute("device.id", event["device_id"])
        span.set_attribute("user.name", event["user"])

        with tracer.start_as_current_span("preprocess_features"):
            # Features are preprocessed inside detect_anomaly
            pass

        with tracer.start_as_current_span("detect_anomaly"):
            anomaly_score, is_anomaly_predicted = detect_anomaly(event)

        span.set_attribute("anomaly.score", anomaly_score)
        span.set_attribute("anomaly.predicted", is_anomaly_predicted)
        span.set_attribute("anomaly.ground_truth", event.get("is_anomaly_ground_truth", False))

        record = {
            "event_id": event_id,
            "timestamp": event["timestamp"],
            "user": event["user"],
            "device_id": event["device_id"],
            "url": event["url"],
            "url_length": event["url_length"],
            "dot_count": event["dot_count"],
            "protocol": event["protocol"],
            "hour": event["hour"],
            "anomaly_score": anomaly_score,
            "is_anomaly_predicted": is_anomaly_predicted,
            "is_anomaly_ground_truth": event.get("is_anomaly_ground_truth", False)
        }

        with tracer.start_as_current_span("write_tsv"):
            writer.writerow(record)
            tsv_file.flush()

        print(
            f"Processed {event_id} → Score: {anomaly_score:.3f}, Predicted: {is_anomaly_predicted}, Ground Truth: {record['is_anomaly_ground_truth']}"
        )